# Predicción de Series Temporales: MLP vs LSTM
## Dataset: USD/CNY Exchange Rate

---

### Mejoras incorporadas en esta versión

| Mejora | Descripción |
|---|---|
| División 70/15/15 | Train / Validation / Test para MLP y LSTM |
| Early Stopping | Detiene el entrenamiento cuando `val_loss` deja de mejorar |
| `val_loss` y `acc_val` | Se registran y grafican curvas de validación |
| LSTM con ventanas | Mismo dataset y ventanas que el MLP para comparación justa |
| Normalización revisada | Se verifica que `inverse_transform` recupere los valores originales |
| Predicción autorregresiva | Uso del modelo de forma iterativa para predecir múltiples pasos |
| Pronóstico 20–30 días | Predicción futura usando el último estado conocido |
| Gráficas de ventanas | Visualización de ventanas temporales de entrada |
| Dataset completo | Se usa el total de datos; se muestra comparativa train+total |

---

## Paso 1 — Importar librerías

| Librería | Uso |
|---|---|
| `pandas` | Carga y manipulación del CSV |
| `numpy` | Operaciones numéricas y ventanas deslizantes |
| `matplotlib` | Gráficas de series, predicciones y curvas |
| `torch / nn` | Definición y entrenamiento de MLP y LSTM |
| `sklearn` | Normalización MinMaxScaler y métricas de regresión |
| `copy` | Copia de pesos del mejor modelo (early stopping) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm import tqdm

# ── Reproducibilidad ─────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✔ Dispositivo: {device}')
print(f'✔ PyTorch: {torch.__version__}')


## Paso 2 — Cargar el dataset

El dataset contiene el tipo de cambio **USD/CNY** con columnas: `Date`, `Open`, `High`, `Low`, `Close`, `Adj Close`, `CNY_pct_change`.

Se usa el **total de datos disponibles** sin filtrar filas innecesariamente.

In [ ]:
df = pd.read_csv("archive/Usd-PKRINRCNY.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ── Usar TODAS las filas disponibles ─────────────────────────────────────────
print(f'📊 Total de registros: {len(df)}')
print(f'📅 Rango: {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'\nColumnas: {list(df.columns)}')
print(f'\nValores nulos:\n{df.isnull().sum()}')
df.head(10)

## Paso 3 — Visualización de la serie temporal completa

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Precio de cierre
axes[0].plot(df['Date'], df['Close'], color='steelblue', linewidth=1.5, label='Close')
axes[0].fill_between(df['Date'], df['Low'], df['High'], alpha=0.15, color='steelblue', label='High-Low')
axes[0].set_ylabel('Tipo de Cambio USD/CNY', fontsize=11)
axes[0].set_title('Serie Temporal Completa — USD/CNY Exchange Rate', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Variación porcentual
axes[1].bar(df['Date'], df['CNY_pct_change'], color='darkorange', alpha=0.6, width=1, label='% Cambio diario')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel('Variación (%)', fontsize=11)
axes[1].set_xlabel('Fecha', fontsize=11)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('serie_completa.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'✔ Total de puntos graficados: {len(df)}')

## Paso 4 — Selección de features y normalización

### Verificación de la normalización

Un paso crítico es confirmar que `inverse_transform` recupera exactamente los valores originales. Si hay discrepancias, el error reportado en USDT/CNY sería incorrecto.

In [ ]:
# ── Features ─────────────────────────────────────────────────────────────────
FEATURE_COLS = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'CNY_pct_change']
TARGET_COL_IDX = 3  # índice de 'Close' en FEATURE_COLS

data = df[FEATURE_COLS].values.astype(np.float32)
dates_all = df['Date'].values

print(f'Shape del dataset completo: {data.shape}')
print(f'Features: {FEATURE_COLS}')
print(f'Variable objetivo: Close (índice {TARGET_COL_IDX})')

# ── Normalización: fit sobre TODO el dataset ─────────────────────────────────
# (también se mostrará la versión fit solo en train más adelante)
scaler_full = MinMaxScaler(feature_range=(0, 1))
data_scaled_full = scaler_full.fit_transform(data)

# ── VERIFICACIÓN: inverse_transform recupera los valores originales ───────────
data_recovered = scaler_full.inverse_transform(data_scaled_full)
max_diff = np.abs(data - data_recovered).max()
print(f'\n✅ Verificación normalización:')
print(f'   Diferencia máxima (original vs recuperado): {max_diff:.2e}')
print(f'   Valores coinciden: {max_diff < 1e-4}')

# Ejemplo con los primeros 3 valores de Close
print(f'\n   Close original [0:3]:   {data[:3, TARGET_COL_IDX]}')
print(f'   Close recuperado [0:3]: {data_recovered[:3, TARGET_COL_IDX].round(6)}')

# ── Scaler solo para Close (desnormalizar predicciones) ──────────────────────
close_scaler = MinMaxScaler(feature_range=(0, 1))
close_scaler.fit(data[:, TARGET_COL_IDX:TARGET_COL_IDX+1])

## Paso 5 — Ventanas deslizantes

Se crean ventanas de `WINDOW = 10` días para predecir el precio `Close` del día siguiente.

**Diferencia clave:**
- **MLP**: recibe la ventana *aplanada* → vector de `WINDOW × n_features`
- **LSTM**: recibe la ventana con estructura temporal → tensor `(WINDOW, n_features)`

In [ ]:
WINDOW = 10  # días de historia para predecir el siguiente
N_FEATURES = len(FEATURE_COLS)

def create_sequences(data_scaled, window, target_idx):
    """
    Genera pares (X, y) usando ventana deslizante.
    
    Parámetros:
    -----------
    data_scaled : np.ndarray (T, n_features) — datos normalizados
    window      : int — tamaño de la ventana de entrada
    target_idx  : int — índice de la columna objetivo en y
    
    Retorna:
    --------
    X : np.ndarray (N, window, n_features) — entradas
    y : np.ndarray (N,)                    — Close del paso siguiente
    """
    X, y = [], []
    for i in range(len(data_scaled) - window):
        X.append(data_scaled[i : i + window])          # ventana
        y.append(data_scaled[i + window, target_idx])  # Close siguiente
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_seq, y_seq = create_sequences(data_scaled_full, WINDOW, TARGET_COL_IDX)
dates_seq = dates_all[WINDOW:]  # fechas correspondientes a cada y

print(f'X shape (secuencias): {X_seq.shape}  → (N, window, features)')
print(f'y shape (objetivos):  {y_seq.shape}')
print(f'Total secuencias generadas del dataset completo: {len(X_seq)}')

### Visualización de ventanas temporales

Se usa `plot_series` del cuadernillo de referencia para mostrar **15 ventanas** distribuidas a lo largo del dataset (grid 3×5). Cada subgráfico muestra:

- **Línea azul (`.-`)**: los `WINDOW` días de historia (entrada del modelo)
- **Cruz azul (`bx`)**: el valor real a predecir (objetivo)

Luego se reutiliza la misma función pasando `y_pred` para comparar Real vs MLP y Real vs LSTM en el mismo formato.


In [ ]:
# ── plot_series: función del cuadernillo de referencia ─────────────────────
# Grid fijo 3×5 = 15 ventanas. Muestra historia (línea), real (bx) y pred (ro).
# Adaptación: series → X con solo la columna Close, y/y_pred → shape (N,1)

def plot_series(series, y=None, y_pred=None, y_pred_std=None,
                x_label="$t$ (días)", y_label="$x$", title=""):
    """
    Visualiza ventanas temporales en un grid 3×5 (15 subgráficos).

    Parámetros:
    -----------
    series     : np.ndarray (N, window)  — historia de cada ventana (Close normalizado)
    y          : np.ndarray (N, 1)       — valor real objetivo
    y_pred     : np.ndarray (N, 1)       — predicción del modelo  (opcional)
    y_pred_std : np.ndarray (N, 1)       — desviación estándar    (opcional, IC)
    title      : str                     — título del gráfico

    Nota:
    -----
    - La línea horizontal en y=0 y el eje y=[-1,1] son correctos porque
      los datos están normalizados con MinMaxScaler en [0,1], centrados
      visualmente. Se ajusta el eje al rango real de los datos.
    """
    r, c = 3, 5
    n_plots = r * c        # 15 ventanas
    # Seleccionar 15 índices distribuidos uniformemente en el dataset
    indices = np.linspace(0, len(series) - 1, n_plots, dtype=int)

    fig, axes = plt.subplots(nrows=r, ncols=c, sharey=True, sharex=True,
                              figsize=(20, 10))
    if title:
        fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)

    for row in range(r):
        for col in range(c):
            plt.sca(axes[row][col])
            ix = indices[col + row * c]   # índice real en el dataset

            # Historia de la ventana
            plt.plot(series[ix, :], ".-", color="steelblue", linewidth=1.5,
                     markersize=4, label="Historia")

            # Valor real (objetivo)
            if y is not None:
                plt.plot(
                    range(len(series[ix, :]),
                          len(series[ix, :]) + len(np.atleast_1d(y[ix]))),
                    np.atleast_1d(y[ix]),
                    "bx", markersize=12, markeredgewidth=2, label="Real"
                )

            # Predicción del modelo
            if y_pred is not None:
                plt.plot(
                    range(len(series[ix, :]),
                          len(series[ix, :]) + len(np.atleast_1d(y_pred[ix]))),
                    np.atleast_1d(y_pred[ix]),
                    "ro", markersize=8, label="Pred"
                )

            # Bandas de incertidumbre
            if y_pred_std is not None:
                t_pred = range(len(series[ix, :]),
                               len(series[ix, :]) + len(np.atleast_1d(y_pred[ix])))
                plt.plot(t_pred,
                         np.atleast_1d(y_pred[ix]) + np.atleast_1d(y_pred_std[ix]),
                         "g--", linewidth=1)
                plt.plot(t_pred,
                         np.atleast_1d(y_pred[ix]) - np.atleast_1d(y_pred_std[ix]),
                         "g--", linewidth=1)

            plt.grid(True, alpha=0.4)
            plt.hlines(0, 0, WINDOW + 2, linewidth=0.8, colors='gray', linestyles='--')

            # Eje x ajustado a la ventana + 1 paso objetivo
            y_min = series[ix, :].min() - 0.05
            y_max = series[ix, :].max() + 0.05
            plt.axis([0, WINDOW + 2, y_min, y_max])

            # Fecha del objetivo
            fecha = pd.Timestamp(dates_seq[ix]).strftime('%Y-%m-%d')
            plt.title(f'#{ix}  {fecha}', fontsize=8)

            if x_label and row == r - 1:
                plt.xlabel(x_label, fontsize=13)
            if y_label and col == 0:
                plt.ylabel(y_label, fontsize=13, rotation=0, labelpad=20)

    plt.tight_layout()
    plt.savefig(f'ventanas_{title.replace(" ", "_").replace("/", "-")[:30]}.png',
                dpi=110, bbox_inches='tight')
    plt.show()


# ── Preparar arrays para plot_series ─────────────────────────────────────────
# series: solo la columna Close de cada ventana → (N, WINDOW)
series_close = X_seq[:, :, TARGET_COL_IDX]          # (N, WINDOW) — normalizado
y_1d         = y_seq.reshape(-1, 1)                  # (N, 1)      — Close siguiente

# ── 1. Solo historia + real ───────────────────────────────────────────────────
print("Gráfica 1 — Ventanas temporales (historia + valor real):")
plot_series(series_close, y=y_1d,
            title="Ventanas Temporales — Historia + Real")


In [ ]:
# ── 2. Real vs MLP (usando plot_series con y_pred) ───────────────────────────
# Necesita que mlp_model esté entrenado — ejecutar después del Paso 8
print("Gráfica 2 — Real vs MLP por ventana:")

mlp_model.eval()
with torch.no_grad():
    # Predecir sobre TODAS las secuencias (X_flat = X_seq aplanado)
    preds_all_mlp = mlp_model(
        torch.tensor(X_flat, dtype=torch.float32).to(device)
    ).cpu().numpy().reshape(-1, 1)   # (N, 1)

plot_series(series_close, y=y_1d, y_pred=preds_all_mlp,
            title="Real vs MLP")

# ── 3. Real vs LSTM (usando plot_series con y_pred) ──────────────────────────
print("Gráfica 3 — Real vs LSTM por ventana:")

lstm_model.eval()
with torch.no_grad():
    preds_all_lstm = lstm_model(
        torch.tensor(X_seq, dtype=torch.float32).to(device)
    ).cpu().numpy().reshape(-1, 1)   # (N, 1)

plot_series(series_close, y=y_1d, y_pred=preds_all_lstm,
            title="Real vs LSTM")


## Paso 6 — División Train / Validation / Test (70 / 15 / 15)

En series temporales **no se mezclan los datos** (no hay shuffle). La división es estrictamente cronológica:

```
├────────── Train (70%) ──────────┤── Val (15%) ──┤── Test (15%) ──┤
t=0                            t=n1            t=n2             t=T
```

Se muestra también el tamaño sobre el dataset total.

In [ ]:
N = len(X_seq)
n_train = int(N * 0.70)
n_val   = int(N * 0.85)  # 70% + 15% = 85%

# ── Arrays para MLP (X aplanado) y LSTM (X con estructura temporal) ──────────
X_flat = X_seq.reshape(N, WINDOW * N_FEATURES)

X_train_mlp, y_train_mlp = X_flat[:n_train],      y_seq[:n_train]
X_val_mlp,   y_val_mlp   = X_flat[n_train:n_val], y_seq[n_train:n_val]
X_test_mlp,  y_test_mlp  = X_flat[n_val:],        y_seq[n_val:]

X_train_lstm, y_train_lstm = X_seq[:n_train],      y_seq[:n_train]
X_val_lstm,   y_val_lstm   = X_seq[n_train:n_val], y_seq[n_train:n_val]
X_test_lstm,  y_test_lstm  = X_seq[n_val:],        y_seq[n_val:]

dates_train = dates_seq[:n_train]
dates_val   = dates_seq[n_train:n_val]
dates_test  = dates_seq[n_val:]

# ── TimeSeriesDataset ─────────────────────────────────────────────────────────
class TimeSeriesDataset(Dataset):
    """
    Dataset PyTorch para series temporales.
    En modo train=True  devuelve (X, y).
    En modo train=False devuelve solo X  (para predicción sin etiquetas).

    Parámetros:
    -----------
    X     : np.ndarray — ventanas de entrada
    y     : np.ndarray — valores objetivo (opcional en inferencia)
    train : bool       — True durante entrenamiento/validación
    """
    def __init__(self, X, y=None, train=True):
        self.X     = X.astype(np.float32)
        self.y     = y.astype(np.float32) if y is not None else None
        self.train = train

    def __len__(self):
        return len(self.X)

    def __getitem__(self, ix):
        if self.train:
            return torch.from_numpy(self.X[ix]), torch.from_numpy(np.array([self.y[ix]], dtype=np.float32))
        return torch.from_numpy(self.X[ix])


# ── DataLoaders — MLP ─────────────────────────────────────────────────────────
dataloader_mlp = {
    'train': DataLoader(TimeSeriesDataset(X_train_mlp, y_train_mlp),
                        shuffle=False, batch_size=64),
    'eval':  DataLoader(TimeSeriesDataset(X_val_mlp,   y_val_mlp),
                        shuffle=False, batch_size=64),
    'test':  DataLoader(TimeSeriesDataset(X_test_mlp,  train=False),
                        shuffle=False, batch_size=64),
}

# ── DataLoaders — LSTM ────────────────────────────────────────────────────────
dataloader_lstm = {
    'train': DataLoader(TimeSeriesDataset(X_train_lstm, y_train_lstm),
                        shuffle=False, batch_size=64),
    'eval':  DataLoader(TimeSeriesDataset(X_val_lstm,   y_val_lstm),
                        shuffle=False, batch_size=64),
    'test':  DataLoader(TimeSeriesDataset(X_test_lstm,  train=False),
                        shuffle=False, batch_size=64),
}

print('División del dataset completo:')
print(f'  Total secuencias : {N:>6}')
print(f'  Train (70%)      : {len(X_train_mlp):>6}  [{pd.Timestamp(dates_train[0]).date()} → {pd.Timestamp(dates_train[-1]).date()}]')
print(f'  Validation (15%) : {len(X_val_mlp):>6}  [{pd.Timestamp(dates_val[0]).date()} → {pd.Timestamp(dates_val[-1]).date()}]')
print(f'  Test (15%)       : {len(X_test_mlp):>6}  [{pd.Timestamp(dates_test[0]).date()} → {pd.Timestamp(dates_test[-1]).date()}]')
print(f'\n  dataloader_mlp  keys: {list(dataloader_mlp.keys())}')
print(f'  dataloader_lstm keys: {list(dataloader_lstm.keys())}')


In [ ]:
# ── Visualizar la división sobre la serie ────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))

# Desnormalizar Close para graficar
close_all = close_scaler.inverse_transform(y_seq.reshape(-1, 1)).flatten()

ax.plot(dates_train, close_all[:n_train],  color='steelblue',  linewidth=1.5, label=f'Train ({n_train})')
ax.plot(dates_val,   close_all[n_train:n_val], color='darkorange', linewidth=1.5, label=f'Val ({n_val-n_train})')
ax.plot(dates_test,  close_all[n_val:],    color='green',      linewidth=1.5, label=f'Test ({N-n_val})')

ax.axvline(dates_val[0],  color='darkorange', linestyle='--', alpha=0.7)
ax.axvline(dates_test[0], color='green',      linestyle='--', alpha=0.7)
ax.set_title('División Train / Validation / Test — USD/CNY Close', fontsize=13, fontweight='bold')
ax.set_ylabel('Close (CNY)')
ax.set_xlabel('Fecha')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('division_dataset.png', dpi=120, bbox_inches='tight')
plt.show()

## Paso 7 — Construcción de modelos

### MLP mejorado

Se añade:
- Capa adicional para mayor capacidad
- `BatchNorm1d` para estabilizar el entrenamiento
- `Dropout(0.2)` como regularización

### LSTM

- 2 capas LSTM apiladas (`num_layers=2`)
- Procesa la secuencia temporalmente (no aplana)
- Capa densa final sobre la última salida temporal

In [ ]:
# ── MLP — basado en el cuadernillo de referencia ────────────────────────────
class MLP(nn.Module):
    """
    Perceptrón Multicapa para predicción de series temporales.
    Recibe la ventana APLANADA: (batch, WINDOW * N_FEATURES).

    Arquitectura (adaptada del cuadernillo de referencia):
        Linear(n_in)  → ReLU
        Linear(64)    → ReLU
        Linear(32)    → ReLU
        Linear(n_out)

    n_in  = WINDOW * N_FEATURES  (ventana aplanada)
    n_out = 1                    (Close del siguiente día)
    """
    def __init__(self, n_in=None, n_out=1):
        super().__init__()
        if n_in is None:
            n_in = WINDOW * N_FEATURES
        self.fc1 = nn.Linear(n_in, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, n_out)

    def forward(self, x):
        # Aplanar la ventana → (batch, WINDOW * N_FEATURES)
        x = x.view(x.shape[0], -1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x


# ── LSTM ──────────────────────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    """
    LSTM apilado para predicción de series temporales.
    Recibe la secuencia con estructura: (batch, WINDOW, N_FEATURES).

    Hiperparámetros:
        hidden_size : 64  — neuronas en el estado oculto
        num_layers  : 2   — capas LSTM apiladas
        dropout     : 0.2 — regularización entre capas
    """
    def __init__(self, n_features=None, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        if n_features is None:
            n_features = N_FEATURES
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)       # (batch, WINDOW, hidden_size)
        last   = out[:, -1, :]      # último paso temporal
        return self.fc(last)        # (batch, 1)


# ── Instanciar ────────────────────────────────────────────────────────────────
INPUT_MLP  = WINDOW * N_FEATURES
mlp_model  = MLP(n_in=INPUT_MLP)
lstm_model = LSTMModel(n_features=N_FEATURES)

n_mlp  = sum(p.numel() for p in mlp_model.parameters())
n_lstm = sum(p.numel() for p in lstm_model.parameters())

print(f'MLP  — n_in={INPUT_MLP} | parámetros: {n_mlp:,}')
print(f'LSTM — n_features={N_FEATURES} | parámetros: {n_lstm:,}')
print(f'\nMLP:\n{mlp_model}')
print(f'\nLSTM:\n{lstm_model}')


## Paso 8 — Entrenamiento con Early Stopping y métricas de validación

### Early Stopping
Si la `val_loss` no mejora durante `patience` épocas, el entrenamiento se detiene y se restauran los pesos del mejor epoch. Esto evita el sobreajuste.

### Métricas registradas por época
- `train_loss`: MSE sobre el conjunto de entrenamiento
- `val_loss`: MSE sobre el conjunto de validación
- `val_mae`: MAE sobre validación (proxy de "precisión" en regresión)

### Hiperparámetros de entrenamiento

| Parámetro | Valor | Descripción |
|---|---|---|
| `EPOCHS` | 200 | Máximo de épocas |
| `BATCH_SIZE` | 32 | Muestras por mini-batch |
| `LR` | 1e-3 | Learning rate Adam |
| `PATIENCE` | 20 | Épocas sin mejora para early stop |
| `LR_PATIENCE` | 10 | Épocas sin mejora para reducir LR |

In [ ]:
EPOCHS   = 200
PATIENCE = 20

# ── fit — basado en el cuadernillo de referencia con early stopping ───────────
def fit(model, dataloader, epochs=EPOCHS, patience=PATIENCE, model_name='Model'):
    """
    Entrena el modelo usando tqdm como barra de progreso.
    Salida por consola por época:
        loss 0.00689 val_loss 0.00721: 100%|██████| 200/200

    Parámetros:
    -----------
    model      : nn.Module
    dataloader : dict con claves 'train' y 'eval'
    epochs     : int — máximo de épocas
    patience   : int — early stopping
    model_name : str — nombre para logs

    Retorna:
    --------
    history : dict con listas train_loss, val_loss, val_mae
    """
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )

    history    = {'train_loss': [], 'val_loss': [], 'val_mae': []}
    best_val   = float('inf')
    best_wts   = None
    best_epoch = 1
    no_improve = 0

    bar = tqdm(range(1, epochs + 1), desc=f'{model_name}')

    for epoch in bar:
        # ── TRAIN ────────────────────────────────────────────────────────────
        model.train()
        train_loss = []
        for batch in dataloader['train']:
            X, y = batch
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = model(X)
            loss  = criterion(y_hat, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss.append(loss.item())

        # ── EVAL ─────────────────────────────────────────────────────────────
        model.eval()
        eval_loss = []
        eval_mae  = []
        with torch.no_grad():
            for batch in dataloader['eval']:
                X, y = batch
                X, y = X.to(device), y.to(device)
                y_hat = model(X)
                eval_loss.append(criterion(y_hat, y).item())
                eval_mae.append(torch.mean(torch.abs(y_hat - y)).item())

        avg_tr  = np.mean(train_loss)
        avg_vl  = np.mean(eval_loss)
        avg_mae = np.mean(eval_mae)

        history['train_loss'].append(avg_tr)
        history['val_loss'].append(avg_vl)
        history['val_mae'].append(avg_mae)

        scheduler.step(avg_vl)
        lr_now = optimizer.param_groups[0]['lr']

        # ── Actualizar barra tqdm (formato del cuadernillo de referencia) ────
        bar.set_description(
            f"loss {avg_tr:.5f}  val_loss {avg_vl:.5f}  val_mae {avg_mae:.5f}  lr {lr_now:.2e}"
        )

        # ── Early stopping ───────────────────────────────────────────────────
        if avg_vl < best_val:
            best_val   = avg_vl
            best_epoch = epoch
            best_wts   = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                bar.write(f'  ⏹ Early stopping en época {epoch} — mejor época: {best_epoch} — best val_loss: {best_val:.5f}')
                break

    model.load_state_dict(best_wts)
    bar.write(f'  ✔ [{model_name}] Finalizado. Mejor val_loss: {best_val:.5f} (época {best_epoch})')
    return history


# ── predict — basado en el cuadernillo de referencia ─────────────────────────
def predict(model, dataloader):
    """
    Genera predicciones sobre el DataLoader de test (sin etiquetas).
    Usa model.eval() y torch.no_grad() para inferencia eficiente.

    Retorna:
    --------
    preds : torch.Tensor (N, 1) en el dispositivo activo
    """
    model.eval()
    with torch.no_grad():
        preds = torch.tensor([]).to(device)
        for batch in dataloader:
            X    = batch
            X    = X.to(device)
            pred = model(X)
            preds = torch.cat([preds, pred])
    return preds


print('✔ fit() y predict() definidos')
print(f'  Salida por época: loss X.XXXXX  val_loss X.XXXXX  val_mae X.XXXXX  lr X.XXe-XX')


In [ ]:
# ── Entrenar MLP ─────────────────────────────────────────────────────────────
print('\n' + '═'*72)
print('  MODELO 1: MLP')
print(f'  Input: ventana aplanada ({WINDOW} días × {N_FEATURES} features = {WINDOW*N_FEATURES})')
print(f'  Épocas máx: {EPOCHS} | Batch: 64 | LR: 1e-3 | Patience: {PATIENCE}')
print('═'*72)

mlp_model = MLP(n_in=INPUT_MLP)
history_mlp = fit(mlp_model, dataloader_mlp, epochs=EPOCHS,
                  patience=PATIENCE, model_name='MLP')


In [ ]:
# ── Entrenar LSTM ─────────────────────────────────────────────────────────────
print('\n' + '═'*72)
print('  MODELO 2: LSTM')
print(f'  Input: secuencia ({WINDOW} pasos × {N_FEATURES} features) — orden temporal preservado')
print(f'  Épocas máx: {EPOCHS} | Batch: 64 | LR: 1e-3 | Patience: {PATIENCE}')
print('═'*72)

lstm_model = LSTMModel(n_features=N_FEATURES)
history_lstm = fit(lstm_model, dataloader_lstm, epochs=EPOCHS,
                   patience=PATIENCE, model_name='LSTM')


## Paso 9 — Curvas de aprendizaje (train_loss, val_loss, val_mae)

In [ ]:
def plot_history(history, model_name, color):
    epochs_ran = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Loss
    axes[0].plot(epochs_ran, history['train_loss'], color=color,    label='Train Loss', linewidth=2)
    axes[0].plot(epochs_ran, history['val_loss'],   color=color,    label='Val Loss',   linewidth=2, linestyle='--')
    min_vl_idx = np.argmin(history['val_loss'])
    axes[0].axvline(min_vl_idx + 1, color='gray', linestyle=':', alpha=0.7)
    axes[0].annotate(f'min val={history["val_loss"][min_vl_idx]:.5f}\n(época {min_vl_idx+1})',
                     xy=(min_vl_idx + 1, history['val_loss'][min_vl_idx]),
                     xytext=(min_vl_idx + 3, history['val_loss'][min_vl_idx] * 1.05),
                     fontsize=8, color='gray')
    axes[0].set_title(f'{model_name} — Train vs Val Loss (MSE)', fontweight='bold')
    axes[0].set_xlabel('Época')
    axes[0].set_ylabel('MSE Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # MAE de validación
    axes[1].plot(epochs_ran, history['val_mae'], color='darkorange', label='Val MAE', linewidth=2)
    axes[1].set_title(f'{model_name} — Val MAE (proxy de precisión)', fontweight='bold')
    axes[1].set_xlabel('Época')
    axes[1].set_ylabel('MAE (escala normalizada)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'curvas_{model_name.lower()}.png', dpi=120, bbox_inches='tight')
    plt.show()

plot_history(history_mlp,  'MLP',  'steelblue')
plot_history(history_lstm, 'LSTM', '#2ECC71')

## Paso 10 — Predicción en Test y evaluación de métricas

Las predicciones se **desnormalizan** con `close_scaler.inverse_transform` para obtener valores reales en CNY y calcular métricas interpretables.

In [ ]:
def get_predictions(model, X, is_lstm=False):
    """Genera predicciones en escala normalizada."""
    model.eval()
    with torch.no_grad():
        t = torch.tensor(X, dtype=torch.float32).to(device)
        preds = model(t).cpu().numpy()
    return preds

def print_metrics_block(metrics_by_split, model_name, color_char='─'):
    """
    Imprime un bloque de métricas formateado por split.
    Incluye: MSE, RMSE, MAE, MAPE, R².
    """
    w = 72
    print(f'\n  ┌{color_char*w}┐')
    print(f'  │  📊 MÉTRICAS — {model_name:<{w-16}}│')
    print(f'  ├{color_char*w}┤')
    print(f'  │  {"Split":<8} {"MSE":>12} {"RMSE":>12} {"MAE":>12} {"MAPE %":>10} {"R²":>8}  │')
    print(f'  ├{"─"*w}┤')
    for split, m in metrics_by_split.items():
        best_mark = ' ★' if split == 'Test' else '  '
        print(f'  │  {split:<8} {m["MSE"]:>12.6f} {m["RMSE"]:>12.6f} '
              f'{m["MAE"]:>12.6f} {m["MAPE"]:>10.4f} {m["R2"]:>8.4f}{best_mark} │')
    print(f'  └{color_char*w}┘')

def evaluate_predictions(y_true_scaled, y_pred_scaled, close_scaler, split_name, model_name, silent=False):
    """Desnormaliza y calcula MSE, RMSE, MAE, R2 en escala real."""
    y_true = close_scaler.inverse_transform(y_true_scaled.reshape(-1,1)).flatten()
    y_pred = close_scaler.inverse_transform(y_pred_scaled.reshape(-1,1)).flatten()
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return y_true, y_pred, {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}

# ── Predicciones MLP ─────────────────────────────────────────────────────────
p_train_mlp = get_predictions(mlp_model, X_train_mlp)
p_val_mlp   = get_predictions(mlp_model, X_val_mlp)
p_test_mlp  = get_predictions(mlp_model, X_test_mlp)

_,       _, metrics_train_mlp = evaluate_predictions(y_train_mlp, p_train_mlp, close_scaler, 'Train', 'MLP')
_,       _, metrics_val_mlp   = evaluate_predictions(y_val_mlp,   p_val_mlp,   close_scaler, 'Val',   'MLP')
yt_mlp, yp_mlp, metrics_test_mlp = evaluate_predictions(y_test_mlp, p_test_mlp, close_scaler, 'Test',  'MLP')

print_metrics_block({'Train': metrics_train_mlp,
                     'Val':   metrics_val_mlp,
                     'Test':  metrics_test_mlp}, 'MLP', '═')

# Resumen rápido
best_mlp = min(metrics_val_mlp['RMSE'], metrics_test_mlp['RMSE'])
print(f'  → Val RMSE: {metrics_val_mlp["RMSE"]:.6f} | Val MAE: {metrics_val_mlp["MAE"]:.6f} | Val R²: {metrics_val_mlp["R2"]:.4f}')

# ── Predicciones LSTM ─────────────────────────────────────────────────────────
p_train_lstm = get_predictions(lstm_model, X_train_lstm)
p_val_lstm   = get_predictions(lstm_model, X_val_lstm)
p_test_lstm  = get_predictions(lstm_model, X_test_lstm)

_,         _, metrics_train_lstm = evaluate_predictions(y_train_lstm, p_train_lstm, close_scaler, 'Train', 'LSTM')
_,         _, metrics_val_lstm   = evaluate_predictions(y_val_lstm,   p_val_lstm,   close_scaler, 'Val',   'LSTM')
yt_lstm, yp_lstm, metrics_test_lstm = evaluate_predictions(y_test_lstm, p_test_lstm, close_scaler, 'Test', 'LSTM')

print_metrics_block({'Train': metrics_train_lstm,
                     'Val':   metrics_val_lstm,
                     'Test':  metrics_test_lstm}, 'LSTM', '═')

print(f'  → Val RMSE: {metrics_val_lstm["RMSE"]:.6f} | Val MAE: {metrics_val_lstm["MAE"]:.6f} | Val R²: {metrics_val_lstm["R2"]:.4f}')


In [ ]:
# ── Tabla comparativa final mejorada ─────────────────────────────────────────
rows = []
for name, m_tr, m_vl, m_te in [
    ('MLP',  metrics_train_mlp,  metrics_val_mlp,  metrics_test_mlp),
    ('LSTM', metrics_train_lstm, metrics_val_lstm, metrics_test_lstm),
]:
    for split, m in [('Train', m_tr), ('Val', m_vl), ('Test', m_te)]:
        rows.append({'Modelo': name, 'Split': split,
                     'MSE': m['MSE'], 'RMSE': m['RMSE'],
                     'MAE': m['MAE'], 'R2': m['R2'], 'MAPE(%)': m['MAPE']})

df_metrics = pd.DataFrame(rows)
df_metrics[['MSE','RMSE','MAE','R2','MAPE(%)']] = df_metrics[['MSE','RMSE','MAE','R2','MAPE(%)']].round(6)

print()
print('  ╔' + '═'*76 + '╗')
print('  ║' + '  📋 TABLA COMPARATIVA COMPLETA — MLP vs LSTM'.center(76) + '║')
print('  ╠' + '═'*76 + '╣')
hdr = f"  ║  {'Modelo':<8} {'Split':<7} {'MSE':>12} {'RMSE':>12} {'MAE':>12} {'R²':>8} {'MAPE%':>8}  ║"
print(hdr)
print('  ╠' + '─'*76 + '╣')

mejor_test_rmse = df_metrics[df_metrics['Split']=='Test']['RMSE'].min()

prev_model = None
for _, row in df_metrics.iterrows():
    if prev_model and prev_model != row['Modelo']:
        print('  ╠' + '─'*76 + '╣')
    prev_model = row['Modelo']
    mark = ' ★' if (row['Split']=='Test' and row['RMSE']==mejor_test_rmse) else '  '
    split_color = {'Train': '🔵', 'Val': '🟠', 'Test': '🟢'}[row['Split']]
    print(f"  ║  {row['Modelo']:<8} {split_color}{row['Split']:<6} "
          f"{row['MSE']:>12.6f} {row['RMSE']:>12.6f} "
          f"{row['MAE']:>12.6f} {row['R2']:>8.4f} {row['MAPE(%)']:>8.4f}{mark}  ║")

print('  ╚' + '═'*76 + '╝')
print()
print('  🔵 Train | 🟠 Validation | 🟢 Test | ★ Mejor RMSE en Test')

# Guardar también como DataFrame
print()
print(df_metrics.to_string(index=False))


## Paso 11 — Visualización de predicciones en Test

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

for ax, (name, yt, yp, color, metrics) in zip(axes, [
    ('MLP',  yt_mlp,  yp_mlp,  'steelblue', metrics_test_mlp),
    ('LSTM', yt_lstm, yp_lstm, '#2ECC71',   metrics_test_lstm),
]):
    n = min(len(dates_test), len(yt))
    ax.plot(dates_test[:n], yt[:n], color='#2C3E50', linewidth=1.5, label='Real', zorder=2)
    ax.plot(dates_test[:n], yp[:n], color=color,     linewidth=1.5, linestyle='--',
            label=f'{name} — RMSE: {metrics["RMSE"]:.5f} | R²: {metrics["R2"]:.4f}', zorder=3)
    ax.fill_between(dates_test[:n], yt[:n], yp[:n], alpha=0.15, color=color)
    ax.set_ylabel('USD/CNY Close', fontsize=11)
    ax.set_title(f'{name} — Predicciones en Test', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

axes[-1].set_xlabel('Fecha', fontsize=11)
fig.suptitle('Predicciones en Conjunto de Test — MLP vs LSTM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('predicciones_test.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Predicción sobre TODO el dataset (train + val + test) ───────────────────
# Muestra cuánto ajusta cada modelo en el total de los datos

p_all_mlp  = get_predictions(mlp_model,  X_flat)
p_all_lstm = get_predictions(lstm_model, X_seq)

y_all_real = close_scaler.inverse_transform(y_seq.reshape(-1,1)).flatten()
y_all_mlp  = close_scaler.inverse_transform(p_all_mlp.reshape(-1,1)).flatten()
y_all_lstm = close_scaler.inverse_transform(p_all_lstm.reshape(-1,1)).flatten()

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
sep_train = pd.Timestamp(dates_val[0])
sep_val   = pd.Timestamp(dates_test[0])

for ax, (name, y_pred_all, color) in zip(axes, [
    ('MLP',  y_all_mlp,  'steelblue'),
    ('LSTM', y_all_lstm, '#2ECC71'),
]):
    ax.plot(dates_seq, y_all_real, color='#2C3E50', linewidth=1, label='Real', zorder=2)
    ax.plot(dates_seq, y_pred_all, color=color, linewidth=1, linestyle='--', label=name, zorder=3)
    ax.axvspan(dates_seq[0],  sep_train, alpha=0.06, color='steelblue', label='Train')
    ax.axvspan(sep_train, sep_val,       alpha=0.06, color='darkorange', label='Val')
    ax.axvspan(sep_val, dates_seq[-1],   alpha=0.06, color='green', label='Test')
    ax.axvline(sep_train, color='darkorange', linestyle='--', linewidth=1)
    ax.axvline(sep_val,   color='green',      linestyle='--', linewidth=1)
    ax.set_title(f'{name} — Predicción sobre el Dataset Completo', fontsize=12, fontweight='bold')
    ax.set_ylabel('USD/CNY Close')
    ax.legend(fontsize=8, ncol=5)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

axes[-1].set_xlabel('Fecha')
fig.suptitle('Predicción sobre Dataset Completo (Train + Val + Test)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('prediccion_completa.png', dpi=120, bbox_inches='tight')
plt.show()

## Paso 12 — Predicción Autorregresiva (Recursive Forecast)

La **predicción autorregresiva** usa el modelo de forma iterativa:
1. Se parte de una ventana de `WINDOW` días reales conocidos.
2. El modelo predice el día $t+1$.
3. Ese valor predicho se agrega a la ventana, desplazando el primero.
4. Se repite el proceso para los siguientes pasos.

```
Ventana inicial:  [d1, d2, ..., d10]  →  predice d11
Nueva ventana:    [d2, d3, ..., d11*] →  predice d12*
Nueva ventana:    [d3, d4, ..., d12*] →  predice d13*
...
```

> **Nota:** cada error se propaga en las predicciones siguientes, por eso el LSTM suele degradarse menos al ser más robusto en secuencias largas.

In [ ]:
def predict_autoregressive(model, seed_window_seq, n_steps, close_scaler,
                           n_features, target_idx, is_lstm=True):
    """
    Predicción autorregresiva: usa predicciones pasadas como entrada.
    
    Parámetros:
    -----------
    model          : nn.Module entrenado
    seed_window_seq: np.ndarray (window, n_features) — ventana inicial normalizada
    n_steps        : int — número de pasos a predecir
    close_scaler   : scaler ajustado solo para Close
    n_features     : int — número de features
    target_idx     : int — índice de Close en features
    is_lstm        : bool — True si el modelo espera (batch, window, features)
    
    Retorna:
    --------
    preds_real : np.ndarray (n_steps,) — predicciones en CNY real
    """
    model.eval()
    current_window = seed_window_seq.copy()  # (window, n_features)
    preds_scaled = []

    with torch.no_grad():
        for step in range(n_steps):
            if is_lstm:
                inp = torch.tensor(current_window, dtype=torch.float32).unsqueeze(0).to(device)
            else:
                inp = torch.tensor(current_window.flatten(), dtype=torch.float32).unsqueeze(0).to(device)
            
            pred_scaled = model(inp).cpu().numpy().flatten()[0]
            preds_scaled.append(pred_scaled)
            
            # Crear nueva fila: copiamos la última fila y actualizamos Close
            new_row = current_window[-1].copy()
            new_row[target_idx] = pred_scaled
            
            # Desplazar ventana (quitar el primero, agregar nuevo al final)
            current_window = np.vstack([current_window[1:], new_row])

    # Desnormalizar
    preds_real = close_scaler.inverse_transform(
        np.array(preds_scaled).reshape(-1, 1)
    ).flatten()
    return preds_real


# ── Evaluar predicción autorregresiva en Test ─────────────────────────────────
# Semilla: última ventana antes del test
seed_window = X_seq[n_train]  # primera ventana del test
N_AUTO = len(X_test_lstm)     # predecir el mismo número que hay en test

auto_mlp  = predict_autoregressive(mlp_model,  seed_window, N_AUTO, close_scaler,
                                   N_FEATURES, TARGET_COL_IDX, is_lstm=False)
auto_lstm = predict_autoregressive(lstm_model, seed_window, N_AUTO, close_scaler,
                                   N_FEATURES, TARGET_COL_IDX, is_lstm=True)

# Métricas autorregresivas
y_test_real = close_scaler.inverse_transform(y_test_lstm.reshape(-1,1)).flatten()
rmse_auto_mlp  = np.sqrt(mean_squared_error(y_test_real, auto_mlp))
rmse_auto_lstm = np.sqrt(mean_squared_error(y_test_real, auto_lstm))
print(f'Predicción autorregresiva en Test:')
print(f'  MLP  RMSE: {rmse_auto_mlp:.5f}')
print(f'  LSTM RMSE: {rmse_auto_lstm:.5f}')
print(f'  (comparar con RMSE directo MLP:{metrics_test_mlp["RMSE"]:.5f} | LSTM:{metrics_test_lstm["RMSE"]:.5f})')

In [ ]:
# ── Gráfica: Predicción directa vs Autorregresiva ────────────────────────────
n = min(len(dates_test), N_AUTO)

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)

for ax, (name, y_direct, y_auto, color) in zip(axes, [
    ('MLP',  yp_mlp,  auto_mlp,  'steelblue'),
    ('LSTM', yp_lstm, auto_lstm, '#2ECC71'),
]):
    ax.plot(dates_test[:n], y_test_real[:n], color='#2C3E50', linewidth=2, label='Real', zorder=3)
    ax.plot(dates_test[:n], y_direct[:n],    color=color, linewidth=1.5,
            linestyle='--', label=f'{name} Directa', zorder=2)
    ax.plot(dates_test[:n], y_auto[:n],      color='#E74C3C', linewidth=1.5,
            linestyle=':', label=f'{name} Autorregresiva', zorder=2)
    ax.set_ylabel('USD/CNY Close', fontsize=11)
    ax.set_title(f'{name} — Predicción Directa vs Autorregresiva', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Fecha', fontsize=11)
fig.suptitle('Comparación: Predicción Directa vs Autorregresiva', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pred_autorregresiva.png', dpi=120, bbox_inches='tight')
plt.show()
print('✔ Nota: La predicción autorregresiva acumula errores con cada paso,\n'
      '        pero sirve para pronósticos a futuro sin datos reales disponibles.')

## Paso 13 — Pronóstico Futuro: 20 a 30 días

Usando la **predicción autorregresiva**, extendemos el pronóstico más allá de los datos disponibles:
- Se parte de la **última ventana conocida** (los 10 últimos días del dataset)
- Se predicen **30 días** hacia adelante
- El intervalo de confianza se estima con varianza acumulada del error de validación

In [ ]:
FORECAST_DAYS = 30  # número de días a predecir hacia el futuro

# Última ventana conocida del dataset completo (últimos WINDOW días)
last_window = data_scaled_full[-WINDOW:]   # (WINDOW, n_features)

# ── Pronóstico futuro con MLP y LSTM ────────────────────────────────────────
future_mlp  = predict_autoregressive(mlp_model,  last_window, FORECAST_DAYS,
                                     close_scaler, N_FEATURES, TARGET_COL_IDX, is_lstm=False)
future_lstm = predict_autoregressive(lstm_model, last_window, FORECAST_DAYS,
                                     close_scaler, N_FEATURES, TARGET_COL_IDX, is_lstm=True)

# Generar fechas futuras (días hábiles)
last_date = df['Date'].max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                              periods=FORECAST_DAYS, freq='B')  # 'B' = Business days

# Intervalo de confianza simple: ±2σ del error de validación
val_errors_lstm = np.abs(
    close_scaler.inverse_transform(y_val_lstm.reshape(-1,1)).flatten() -
    close_scaler.inverse_transform(p_val_lstm.reshape(-1,1)).flatten()
)
sigma = val_errors_lstm.std()
# La incertidumbre crece con cada paso autorregresivo
uncertainty = np.array([sigma * np.sqrt(i + 1) for i in range(FORECAST_DAYS)])

print(f'📅 Último dato conocido: {last_date.date()}')
print(f'📅 Pronóstico: {future_dates[0].date()} → {future_dates[-1].date()}')
print(f'\nPronóstico LSTM (próximos {FORECAST_DAYS} días):')
for i, (d, v) in enumerate(zip(future_dates, future_lstm)):
    print(f'  Día {i+1:>2} [{d.date()}]: {v:.5f} CNY/USD')

In [ ]:
# ── Gráfica de pronóstico futuro ─────────────────────────────────────────────
# Mostrar los últimos 60 días reales + el pronóstico de 30 días
N_HISTORY = 60
hist_dates = df['Date'].values[-N_HISTORY:]
hist_close = df['Close'].values[-N_HISTORY:]

fig, ax = plt.subplots(figsize=(15, 6))

# Datos históricos
ax.plot(hist_dates, hist_close, color='#2C3E50', linewidth=2, label='Histórico real', zorder=3)

# Último punto de conexión
connect_date  = [hist_dates[-1], future_dates[0]]
connect_close = [hist_close[-1], future_mlp[0]]

# Pronóstico MLP
ax.plot(np.concatenate([[hist_dates[-1]], future_dates]),
        np.concatenate([[hist_close[-1]], future_mlp]),
        color='steelblue', linewidth=2, linestyle='--', marker='o', markersize=4,
        label=f'MLP — Pronóstico {FORECAST_DAYS}d', zorder=2)

# Pronóstico LSTM con intervalo de confianza
ax.plot(np.concatenate([[hist_dates[-1]], future_dates]),
        np.concatenate([[hist_close[-1]], future_lstm]),
        color='#2ECC71', linewidth=2.5, linestyle='-', marker='s', markersize=4,
        label=f'LSTM — Pronóstico {FORECAST_DAYS}d', zorder=3)

# Intervalo de confianza del LSTM
ax.fill_between(
    future_dates,
    future_lstm - 2 * uncertainty,
    future_lstm + 2 * uncertainty,
    alpha=0.15, color='#2ECC71', label='IC 95% LSTM (±2σ acumulado)'
)
ax.fill_between(
    future_dates,
    future_lstm - uncertainty,
    future_lstm + uncertainty,
    alpha=0.25, color='#2ECC71', label='IC 68% LSTM (±σ acumulado)'
)

# Línea separadora
ax.axvline(hist_dates[-1], color='gray', linestyle=':', linewidth=1.5)
ax.text(hist_dates[-1], ax.get_ylim()[1] * 0.99, ' ← Datos reales | Pronóstico →',
        fontsize=9, color='gray', va='top')

ax.set_title(f'Pronóstico Futuro — USD/CNY ({FORECAST_DAYS} días hábiles)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Tipo de Cambio USD/CNY')
ax.set_xlabel('Fecha')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig('pronostico_futuro.png', dpi=120, bbox_inches='tight')
plt.show()
print('✔ Pronóstico guardado como pronostico_futuro.png')

In [ ]:
# ── Tabla de pronóstico ──────────────────────────────────────────────────────
df_forecast = pd.DataFrame({
    'Fecha': future_dates,
    'MLP_pred': future_mlp.round(5),
    'LSTM_pred': future_lstm.round(5),
    'LSTM_IC_low':  (future_lstm - 2*uncertainty).round(5),
    'LSTM_IC_high': (future_lstm + 2*uncertainty).round(5),
})
print(f'\n📅 TABLA DE PRONÓSTICO — {FORECAST_DAYS} días futuros')
print(df_forecast.to_string(index=False))

## Paso 14 — Resumen comparativo final

### MLP vs LSTM — Diferencias clave

| Aspecto | MLP | LSTM |
|---|---|---|
| Procesa orden temporal | ❌ No — aplana la ventana | ✅ Sí — paso a paso |
| Memoria interna | ❌ No | ✅ Celda de memoria con puertas |
| Gradiente evanescente | No aplica | ✅ Resuelto por puertas forget/input |
| Robustez autorregresiva | Menor — acumula error rápido | Mayor — filtra ruido con memoria |
| Parámetros | Menos | Más |
| Velocidad | Más rápido | Más lento |

### Validación de normalización
Se verificó que `MinMaxScaler.inverse_transform` recupera los valores originales con diferencia máxima < 1e-4, confirmando que las métricas en CNY son correctas.

In [ ]:
# ── Resumen final ────────────────────────────────────────────────────────────
print('\n' + '★' * 60)
print(' RESUMEN FINAL')
print('★' * 60)
print(f'Dataset: USD/CNY | Total días: {len(df)} | Ventana: {WINDOW} días')
print(f'División: {n_train} train / {n_val-n_train} val / {N-n_val} test  (70/15/15)')
print(f'Features: {FEATURE_COLS}')
print(f'Pronóstico futuro: {FORECAST_DAYS} días')
print()
print(f'{"Modelo":<10} {"Split":<8} {"RMSE":>10} {"MAE":>10} {"R²":>8} {"MAPE%":>8}')
print('-' * 58)
for _, row in df_metrics.iterrows():
    marker = ' ⭐' if row['R2'] == df_metrics.loc[df_metrics['Split']=='Test','R2'].max() and row['Split']=='Test' else ''
    print(f'{row["Modelo"]:<10} {row["Split"]:<8} {row["RMSE"]:>10.5f} {row["MAE"]:>10.5f} {row["R2"]:>8.4f} {row["MAPE(%)"]:>8.4f}{marker}')
print('★' * 60)